In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from geopy.geocoders import Nominatim
import time
from sklearn.preprocessing import LabelEncoder
import json
import requests
import folium

# 임의값으로 만든 데이터 셋

## 도로 데이터 로드

In [37]:
df = pd.read_csv("road_data.csv")
df

,DTCT_UID,DTCT_PD,PLC_LTTD,PLC_LGTD,ROAD_INFO,DSTA_LGLD_INFO,DSTA_ADSTRD_INFO
0,gnsoft1_250714084125_11277,2025-07-14 08:41:25,36.346661,127.353838,대전광역시 서구 월평동 114-4,월평동,월평1동
1,gnsoft1_251106093349_24,2025-11-06 09:33:49,36.390977,127.407906,대전광역시 유성구 문지동 661-3,문지동,전민동
2,gnsoft1_251106093431_32,2025-11-06 09:34:31,36.390877,127.407925,대전광역시 유성구 문지동 661-3,문지동,전민동
3,gnsoft1_251106194824_90,2025-11-06 19:48:24,36.350716,127.368103,대전광역시 서구 갈마동 690,갈마동,갈마1동
4,gnsoft2_250712155250_39377,2025-07-12 15:52:51,36.348692,127.335768,대전광역시 유성구 상대동 556,상대동,온천1동
...,...,...,...,...,...,...,...
1074,yuseong1_250221140321_0,2025-02-21 14:03:22,36.411699,127.377649,대전광역시 유성구 화암동 156-8,화암동,신성동
1075,yuseong1_250224143428_15517,2025-02-24 14:34:28,36.405715,127.368060,대전광역시 유성구 방현동 350,방현동,신성동
1076,yuseong1_250224143512_0,2025-02-24 14:35:13,36.401399,127.362489,대전광역시 유성구 장동 241-5,장동,신성동
1077,yuseong1_250224144720_1513,2025-02-24 14:47:21,36.372210,127.320740,대전광역시 유성구 지족동 976,지족동,노은1동


In [38]:
df.columns

Index(['DTCT_UID', 'DTCT_PD', 'PLC_LTTD', 'PLC_LGTD', 'ROAD_INFO',
       'DSTA_LGLD_INFO', 'DSTA_ADSTRD_INFO'],
      dtype='object')

In [39]:
df.drop(columns=["DSTA_LGLD_INFO", "DSTA_ADSTRD_INFO"], inplace=True)
df

,DTCT_UID,DTCT_PD,PLC_LTTD,PLC_LGTD,ROAD_INFO
0,gnsoft1_250714084125_11277,2025-07-14 08:41:25,36.346661,127.353838,대전광역시 서구 월평동 114-4
1,gnsoft1_251106093349_24,2025-11-06 09:33:49,36.390977,127.407906,대전광역시 유성구 문지동 661-3
2,gnsoft1_251106093431_32,2025-11-06 09:34:31,36.390877,127.407925,대전광역시 유성구 문지동 661-3
3,gnsoft1_251106194824_90,2025-11-06 19:48:24,36.350716,127.368103,대전광역시 서구 갈마동 690
4,gnsoft2_250712155250_39377,2025-07-12 15:52:51,36.348692,127.335768,대전광역시 유성구 상대동 556
...,...,...,...,...,...
1074,yuseong1_250221140321_0,2025-02-21 14:03:22,36.411699,127.377649,대전광역시 유성구 화암동 156-8
1075,yuseong1_250224143428_15517,2025-02-24 14:34:28,36.405715,127.368060,대전광역시 유성구 방현동 350
1076,yuseong1_250224143512_0,2025-02-24 14:35:13,36.401399,127.362489,대전광역시 유성구 장동 241-5
1077,yuseong1_250224144720_1513,2025-02-24 14:47:21,36.372210,127.320740,대전광역시 유성구 지족동 976


In [40]:
df['DTCT_PD'] = pd.to_datetime(df['DTCT_PD'])

# 날짜 분리
df['YEAR'] = df['DTCT_PD'].dt.year
df['MONTH'] = df['DTCT_PD'].dt.month
df['DAY'] = df['DTCT_PD'].dt.day
df['HOUR'] = df['DTCT_PD'].dt.hour

# 요일 (0=월요일, 6=일요일)
df['WEEKDAY'] = df['DTCT_PD'].dt.weekday

# 계절
def get_season(month):
    if month in [12, 1, 2]:
        return 'winter'
    elif month in [3, 4, 5]:
        return 'spring'
    elif month in [6, 7, 8]:
        return 'summer'
    else:
        return 'autumn'

df['SEASON'] = df['MONTH'].apply(get_season)

# 출퇴근 시간대 여부
def commute_flag(hour):
    if 8 <= hour <= 9:        
        return 1
    elif 18 <= hour <= 19:   
        return 1
    else:
        return 0

df['COMMUTE'] = df['HOUR'].apply(commute_flag)

In [41]:
df

,DTCT_UID,DTCT_PD,PLC_LTTD,PLC_LGTD,ROAD_INFO,YEAR,MONTH,DAY,HOUR,WEEKDAY,SEASON,COMMUTE
0,gnsoft1_250714084125_11277,2025-07-14 08:41:25,36.346661,127.353838,대전광역시 서구 월평동 114-4,2025,7,14,8,0,summer,1
1,gnsoft1_251106093349_24,2025-11-06 09:33:49,36.390977,127.407906,대전광역시 유성구 문지동 661-3,2025,11,6,9,3,autumn,1
2,gnsoft1_251106093431_32,2025-11-06 09:34:31,36.390877,127.407925,대전광역시 유성구 문지동 661-3,2025,11,6,9,3,autumn,1
3,gnsoft1_251106194824_90,2025-11-06 19:48:24,36.350716,127.368103,대전광역시 서구 갈마동 690,2025,11,6,19,3,autumn,1
4,gnsoft2_250712155250_39377,2025-07-12 15:52:51,36.348692,127.335768,대전광역시 유성구 상대동 556,2025,7,12,15,5,summer,0
...,...,...,...,...,...,...,...,...,...,...,...,...
1074,yuseong1_250221140321_0,2025-02-21 14:03:22,36.411699,127.377649,대전광역시 유성구 화암동 156-8,2025,2,21,14,4,winter,0
1075,yuseong1_250224143428_15517,2025-02-24 14:34:28,36.405715,127.368060,대전광역시 유성구 방현동 350,2025,2,24,14,0,winter,0
1076,yuseong1_250224143512_0,2025-02-24 14:35:13,36.401399,127.362489,대전광역시 유성구 장동 241-5,2025,2,24,14,0,winter,0
1077,yuseong1_250224144720_1513,2025-02-24 14:47:21,36.372210,127.320740,대전광역시 유성구 지족동 976,2025,2,24,14,0,winter,0


In [42]:
df.to_csv("data_1.csv", index=False)

## rolling count, 복합 파손 여부 생성

In [ ]:
df['damage_type'] = 'pothole'

pothole_df = df.copy()

pothole_df = pothole_df.sort_values(['ROAD_INFO', 'DTCT_PD'])
pothole_df = pothole_df.set_index('DTCT_PD')

pothole_df['COUNT_7D'] = (
    pothole_df.groupby('ROAD_INFO')['DTCT_UID']
    .rolling('7D')
    .count()
    .reset_index(level=0, drop=True)
)

pothole_df['COUNT_30D'] = (
    pothole_df.groupby('ROAD_INFO')['DTCT_UID']
    .rolling('30D')
    .count()
    .reset_index(level=0, drop=True)
)

pothole_df['COUNT_90D'] = (
    pothole_df.groupby('ROAD_INFO')['DTCT_UID']
    .rolling('90D')
    .count()
    .reset_index(level=0, drop=True)
)

pothole_df = pothole_df.reset_index()

df = df.merge(
    pothole_df[['DTCT_UID','COUNT_7D','COUNT_30D','COUNT_90D']],
    on='DTCT_UID',
    how='left'
)

n = len(df)

n_none = int(n * 0.70)   # 균열 없음
n_mid  = int(n * 0.20)   # 보통 균열
n_high = n - (n_none + n_mid)  # 심한 균열(잔여 수)

crack_none = np.zeros(n_none)  # 0‰
crack_mid  = np.random.uniform(1, 10, n_mid)  # 1~10‰
crack_high = np.random.uniform(10, 15, n_high)  # 10~15‰

crack_values = np.concatenate([crack_none, crack_mid, crack_high])
np.random.shuffle(crack_values)

# df에 균열율 컬럼 추가
df['crack_rate'] = crack_values

df['DATE'] = df['DTCT_PD'].dt.date

grouped = df.groupby(['ROAD_INFO','DATE'])['damage_type'].apply(list).reset_index()

grouped['COMPLEX_DAMAGE'] = grouped['damage_type'].apply(
    lambda x: 1 if ('pothole' in x and 'crack' in x) else 0
)

df = df.merge(
    grouped[['ROAD_INFO','DATE','COMPLEX_DAMAGE']],
    on=['ROAD_INFO','DATE'],
    how='left'
)

In [44]:
# 1) 포트홀 유형 (전부 1)
df['type_pothole'] = 1

# 2) 균열 유형 (crack_rate > 0 이면 1, 아니면 0)
df['type_crack'] = (df['crack_rate'] > 0).astype(int)

In [45]:
df

,DTCT_UID,DTCT_PD,PLC_LTTD,PLC_LGTD,ROAD_INFO,YEAR,MONTH,DAY,HOUR,WEEKDAY,...,COMMUTE,damage_type,COUNT_7D,COUNT_30D,COUNT_90D,crack_rate,DATE,COMPLEX_DAMAGE,type_pothole,type_crack
0,gnsoft1_250714084125_11277,2025-07-14 08:41:25,36.346661,127.353838,대전광역시 서구 월평동 114-4,2025,7,14,8,0,...,1,pothole,1.0,1.0,1.0,0.0,2025-07-14,0,1,0
1,gnsoft1_251106093349_24,2025-11-06 09:33:49,36.390977,127.407906,대전광역시 유성구 문지동 661-3,2025,11,6,9,3,...,1,pothole,1.0,1.0,1.0,0.0,2025-11-06,0,1,0
2,gnsoft1_251106093431_32,2025-11-06 09:34:31,36.390877,127.407925,대전광역시 유성구 문지동 661-3,2025,11,6,9,3,...,1,pothole,2.0,2.0,2.0,0.0,2025-11-06,0,1,0
3,gnsoft1_251106194824_90,2025-11-06 19:48:24,36.350716,127.368103,대전광역시 서구 갈마동 690,2025,11,6,19,3,...,1,pothole,1.0,1.0,1.0,0.0,2025-11-06,0,1,0
4,gnsoft2_250712155250_39377,2025-07-12 15:52:51,36.348692,127.335768,대전광역시 유성구 상대동 556,2025,7,12,15,5,...,0,pothole,1.0,1.0,1.0,0.0,2025-07-12,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1074,yuseong1_250221140321_0,2025-02-21 14:03:22,36.411699,127.377649,대전광역시 유성구 화암동 156-8,2025,2,21,14,4,...,0,pothole,1.0,1.0,1.0,0.0,2025-02-21,0,1,0
1075,yuseong1_250224143428_15517,2025-02-24 14:34:28,36.405715,127.368060,대전광역시 유성구 방현동 350,2025,2,24,14,0,...,0,pothole,1.0,1.0,1.0,0.0,2025-02-24,0,1,0
1076,yuseong1_250224143512_0,2025-02-24 14:35:13,36.401399,127.362489,대전광역시 유성구 장동 241-5,2025,2,24,14,0,...,0,pothole,1.0,1.0,1.0,0.0,2025-02-24,0,1,0
1077,yuseong1_250224144720_1513,2025-02-24 14:47:21,36.372210,127.320740,대전광역시 유성구 지족동 976,2025,2,24,14,0,...,0,pothole,1.0,1.0,1.0,0.0,2025-02-24,0,1,0


In [46]:
df['complex_damage'] = ((df['type_pothole'] == 1) & (df['type_crack'] == 1)).astype(int)
df

,DTCT_UID,DTCT_PD,PLC_LTTD,PLC_LGTD,ROAD_INFO,YEAR,MONTH,DAY,HOUR,WEEKDAY,...,damage_type,COUNT_7D,COUNT_30D,COUNT_90D,crack_rate,DATE,COMPLEX_DAMAGE,type_pothole,type_crack,complex_damage
0,gnsoft1_250714084125_11277,2025-07-14 08:41:25,36.346661,127.353838,대전광역시 서구 월평동 114-4,2025,7,14,8,0,...,pothole,1.0,1.0,1.0,0.0,2025-07-14,0,1,0,0
1,gnsoft1_251106093349_24,2025-11-06 09:33:49,36.390977,127.407906,대전광역시 유성구 문지동 661-3,2025,11,6,9,3,...,pothole,1.0,1.0,1.0,0.0,2025-11-06,0,1,0,0
2,gnsoft1_251106093431_32,2025-11-06 09:34:31,36.390877,127.407925,대전광역시 유성구 문지동 661-3,2025,11,6,9,3,...,pothole,2.0,2.0,2.0,0.0,2025-11-06,0,1,0,0
3,gnsoft1_251106194824_90,2025-11-06 19:48:24,36.350716,127.368103,대전광역시 서구 갈마동 690,2025,11,6,19,3,...,pothole,1.0,1.0,1.0,0.0,2025-11-06,0,1,0,0
4,gnsoft2_250712155250_39377,2025-07-12 15:52:51,36.348692,127.335768,대전광역시 유성구 상대동 556,2025,7,12,15,5,...,pothole,1.0,1.0,1.0,0.0,2025-07-12,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1074,yuseong1_250221140321_0,2025-02-21 14:03:22,36.411699,127.377649,대전광역시 유성구 화암동 156-8,2025,2,21,14,4,...,pothole,1.0,1.0,1.0,0.0,2025-02-21,0,1,0,0
1075,yuseong1_250224143428_15517,2025-02-24 14:34:28,36.405715,127.368060,대전광역시 유성구 방현동 350,2025,2,24,14,0,...,pothole,1.0,1.0,1.0,0.0,2025-02-24,0,1,0,0
1076,yuseong1_250224143512_0,2025-02-24 14:35:13,36.401399,127.362489,대전광역시 유성구 장동 241-5,2025,2,24,14,0,...,pothole,1.0,1.0,1.0,0.0,2025-02-24,0,1,0,0
1077,yuseong1_250224144720_1513,2025-02-24 14:47:21,36.372210,127.320740,대전광역시 유성구 지족동 976,2025,2,24,14,0,...,pothole,1.0,1.0,1.0,0.0,2025-02-24,0,1,0,0


In [47]:
df.to_csv("data_2.csv", index=False, encoding='utf-8-sig')

## 기상청 데이터

In [49]:
df = pd.read_csv("data_2.csv")

df

,DTCT_UID,PLC_LTTD,PLC_LGTD,ROAD_INFO,YEAR,MONTH,DAY,HOUR,WEEKDAY,SEASON,COMMUTE,COUNT_7D,COUNT_30D,COUNT_90D,crack_rate,DATE,type_pothole,type_crack,complex_damage
0,gnsoft1_250714084125_11277,36.346661,127.353838,대전광역시 서구 월평동 114-4,2025,7,14,8,0,summer,1,1,1,1,0.0,2025-07-14,1,0,0
1,gnsoft1_251106093349_24,36.390977,127.407906,대전광역시 유성구 문지동 661-3,2025,11,6,9,3,autumn,1,1,1,1,0.0,2025-11-06,1,0,0
2,gnsoft1_251106093431_32,36.390877,127.407925,대전광역시 유성구 문지동 661-3,2025,11,6,9,3,autumn,1,2,2,2,0.0,2025-11-06,1,0,0
3,gnsoft1_251106194824_90,36.350716,127.368103,대전광역시 서구 갈마동 690,2025,11,6,19,3,autumn,1,1,1,1,0.0,2025-11-06,1,0,0
4,gnsoft2_250712155250_39377,36.348692,127.335768,대전광역시 유성구 상대동 556,2025,7,12,15,5,summer,0,1,1,1,0.0,2025-07-12,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1074,yuseong1_250221140321_0,36.411699,127.377649,대전광역시 유성구 화암동 156-8,2025,2,21,14,4,winter,0,1,1,1,0.0,2025-02-21,1,0,0
1075,yuseong1_250224143428_15517,36.405715,127.368060,대전광역시 유성구 방현동 350,2025,2,24,14,0,winter,0,1,1,1,0.0,2025-02-24,1,0,0
1076,yuseong1_250224143512_0,36.401399,127.362489,대전광역시 유성구 장동 241-5,2025,2,24,14,0,winter,0,1,1,1,0.0,2025-02-24,1,0,0
1077,yuseong1_250224144720_1513,36.372210,127.320740,대전광역시 유성구 지족동 976,2025,2,24,14,0,winter,0,1,1,1,0.0,2025-02-24,1,0,0


In [2]:
import requests
params = {
    'inf': 'SFC',
    'stn': '',
    'tm': '20251121',  # 임의 날짜
    'help': '0',
    'authKey': '_FW6tOhdRS2VurToXQUtMw'
}
resp = requests.get("https://apihub.kma.go.kr/api/typ01/url/stn_inf.php", params=params)
print(resp.text)


#START7777
# STN           LON           LAT STN_SP          HT     HT_PA     HT_TA     HT_WD     HT_RN STN STN_KO               STN_EN               FCT_ID   LAW_ID    BASIN
#  ID         degee        degree --------         m         m         m         m         m  AD -------------------- -------------------- -------- ---------- ----
   90  128.56473000   38.25085000 35100        17.53     18.73      1.70     10.00      1.40  90 속초                 Sokcho               11D20401 5182033035 ----
   93  127.75443000   37.94738000 31201        95.78     96.78      1.50     10.00      1.40 101 북춘천               Bukchuncheon         11D10301 5111025024 ----
   95  127.30420000   38.14787000 31110       155.48    156.98      1.80     13.00      1.50 101 철원                 Cheorwon             11D10101 5178025624 ----
   98  127.06070000   37.90188000 22200       115.62    116.74      1.70     10.00      1.00 119 동두천               Dongducheon          11B20401 4125010300 ----
   99  126.7664

In [3]:
import requests

params = {
    'inf': 'SFC',
    'stn': '',
    'tm': '20251121',
    'help': '0',
    'authKey': '_FW6tOhdRS2VurToXQUtMw'
}

resp = requests.get("https://apihub.kma.go.kr/api/typ01/url/stn_inf.php", params=params)

# "서울" 포함된 줄만 출력
for line in resp.text.splitlines():
    if "서울" in line:
        print(line)


  108  126.96580000   37.57142000 11101        85.67     86.67      1.50     10.00      0.50 108 서울                 Seoul                11B10101 1111017800 ----


In [51]:
authKey = "_FW6tOhdRS2VurToXQUtMw"
stn = 133  # 대전 지점번호

# 1) 날짜 범위 추출
df['DATE'] = pd.to_datetime(df['DATE'])
start_date = df['DATE'].min().strftime("%Y%m%d")
end_date   = df['DATE'].max().strftime("%Y%m%d")

print("조회 범위:", start_date, "~", end_date)

# 2) 기상청 기간조회 API 요청
url = "https://apihub.kma.go.kr/api/typ01/url/kma_sfcdd3.php"

params = {
    "tm1": start_date,
    "tm2": end_date,
    "stn": stn,
    "help": 0,
    "authKey": authKey
}

resp = requests.get(url, params=params).text.strip().splitlines()

# 3) 주석 제거 + 공백 split
data_lines = [line for line in resp if not line.startswith('#') and line.strip() != ""]
rows = [line.split() for line in data_lines]

df_weather_raw = pd.DataFrame(rows)

# 4) 날짜 컬럼 변환
df_weather_raw['DATE'] = pd.to_datetime(df_weather_raw[0], format="%Y%m%d", errors='coerce')

# 5) 최소셋 변수 index 매핑
#    (기상청 일자료 sfcdd3 고정 구조 기준)
colmap = {
    "TA_AVG": 9,
    "TA_MAX": 10,
    "TA_MIN": 12,
    "RN_DAY": 36,
    "RN_DUR": 38,
    "RN_D99": 37,
    "SD_DAY": 39,
    "SD_NEW": 40,
    "SD_MAX": 41,
    "HM_AVG": 16,
    "SI_DAY": 32,
    "SS_DAY": 28
}

# 6) 필요한 값만 추출
df_weather = pd.DataFrame({"DATE": df_weather_raw["DATE"]})

for key, idx in colmap.items():
    df_weather[key] = pd.to_numeric(df_weather_raw[idx], errors='coerce')

# 7) 원본 df와 날짜 기준 merge
df = df.merge(df_weather, on="DATE", how="left")

조회 범위: 20240119 ~ 20251111


| 변수     | 의미           |
| ------ | ------------ |
| TA_AVG | 일 평균기온       |
| TA_MAX | 일 최고기온       |
| TA_MIN | 일 최저기온       |
| RN_DAY | 일 강수량        |
| RN_D99 | 09–09 기준 강수량 |
| RN_DUR | 강수 지속시간      |
| SD_DAY | 일 최심적설       |
| SD_NEW | 신적설          |
| SD_MAX | 최심 적설 시각 코드  |
| HM_AVG | 일 평균 습도      |
| SI_DAY | 일사량          |
| SS_DAY | 일조시간         |


✔ 온도 (TA_AVG, MAX, MIN)

동결–융해 반복 패턴 파악 → 포트홀 원인 1순위

✔ 강수 (RN_DAY, RN_DUR)

아스팔트 하층에 물이 고이면 균열·박리 발생

RN_DUR은 특히 지속 시간 기반 시나리오 피처로 매우 중요

✔ 적설 (SD_DAY, SD_NEW)

제설 → 염화칼슘 → 표층 손상

녹은 뒤 다시 얼면 immediate pothole formation

✔ 습도 (HM_AVG)

건조 속도 감소 → 물 잔류

✔ 일사/일조 (SI_DAY, SS_DAY)

건조 속도 증가

표면 열화와 강도 감소에 영향

In [52]:
df

,DTCT_UID,PLC_LTTD,PLC_LGTD,ROAD_INFO,YEAR,MONTH,DAY,HOUR,WEEKDAY,SEASON,...,TA_MIN,RN_DAY,RN_DUR,RN_D99,SD_DAY,SD_NEW,SD_MAX,HM_AVG,SI_DAY,SS_DAY
0,gnsoft1_250714084125_11277,36.346661,127.353838,대전광역시 서구 월평동 114-4,2025,7,14,8,0,summer,...,1436,3.02,20.9,1200,22.9,2.33,18.3,29.4,4.0,3
1,gnsoft1_251106093349_24,36.390977,127.407906,대전광역시 유성구 문지동 661-3,2025,11,6,9,3,autumn,...,1438,2.13,-9.0,1200,-9.0,-9.00,-9.0,10.8,6.4,838
2,gnsoft1_251106093431_32,36.390877,127.407925,대전광역시 유성구 문지동 661-3,2025,11,6,9,3,autumn,...,1438,2.13,-9.0,1200,-9.0,-9.00,-9.0,10.8,6.4,838
3,gnsoft1_251106194824_90,36.350716,127.368103,대전광역시 서구 갈마동 690,2025,11,6,19,3,autumn,...,1438,2.13,-9.0,1200,-9.0,-9.00,-9.0,10.8,6.4,838
4,gnsoft2_250712155250_39377,36.348692,127.335768,대전광역시 유성구 상대동 556,2025,7,12,15,5,summer,...,1550,3.54,-9.0,1200,-9.0,-9.00,-9.0,37.8,13.9,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1074,yuseong1_250221140321_0,36.411699,127.377649,대전광역시 유성구 화암동 156-8,2025,2,21,14,4,winter,...,1452,2.75,-9.0,1200,0.4,-9.00,-9.0,1.5,10.1,836
1075,yuseong1_250224143428_15517,36.405715,127.368060,대전광역시 유성구 방현동 350,2025,2,24,14,0,winter,...,1434,2.82,-9.0,1200,-9.0,-9.00,-9.0,1.4,10.3,926
1076,yuseong1_250224143512_0,36.401399,127.362489,대전광역시 유성구 장동 241-5,2025,2,24,14,0,winter,...,1434,2.82,-9.0,1200,-9.0,-9.00,-9.0,1.4,10.3,926
1077,yuseong1_250224144720_1513,36.372210,127.320740,대전광역시 유성구 지족동 976,2025,2,24,14,0,winter,...,1434,2.82,-9.0,1200,-9.0,-9.00,-9.0,1.4,10.3,926


In [53]:
df.to_csv("data_3.csv", index=False, encoding='utf-8-sig')

## 교통량(임의값)

In [62]:
df = pd.read_csv("data_3.csv")
df

,DTCT_UID,PLC_LTTD,PLC_LGTD,ROAD_INFO,YEAR,MONTH,DAY,HOUR,WEEKDAY,SEASON,...,TA_MIN,RN_DAY,RN_DUR,RN_D99,SD_DAY,SD_NEW,SD_MAX,HM_AVG,SI_DAY,SS_DAY
0,gnsoft1_250714084125_11277,36.346661,127.353838,대전광역시 서구 월평동 114-4,2025,7,14,8,0,summer,...,1436,3.02,20.9,1200,22.9,2.33,18.3,29.4,4.0,3
1,gnsoft1_251106093349_24,36.390977,127.407906,대전광역시 유성구 문지동 661-3,2025,11,6,9,3,autumn,...,1438,2.13,-9.0,1200,-9.0,-9.00,-9.0,10.8,6.4,838
2,gnsoft1_251106093431_32,36.390877,127.407925,대전광역시 유성구 문지동 661-3,2025,11,6,9,3,autumn,...,1438,2.13,-9.0,1200,-9.0,-9.00,-9.0,10.8,6.4,838
3,gnsoft1_251106194824_90,36.350716,127.368103,대전광역시 서구 갈마동 690,2025,11,6,19,3,autumn,...,1438,2.13,-9.0,1200,-9.0,-9.00,-9.0,10.8,6.4,838
4,gnsoft2_250712155250_39377,36.348692,127.335768,대전광역시 유성구 상대동 556,2025,7,12,15,5,summer,...,1550,3.54,-9.0,1200,-9.0,-9.00,-9.0,37.8,13.9,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1074,yuseong1_250221140321_0,36.411699,127.377649,대전광역시 유성구 화암동 156-8,2025,2,21,14,4,winter,...,1452,2.75,-9.0,1200,0.4,-9.00,-9.0,1.5,10.1,836
1075,yuseong1_250224143428_15517,36.405715,127.368060,대전광역시 유성구 방현동 350,2025,2,24,14,0,winter,...,1434,2.82,-9.0,1200,-9.0,-9.00,-9.0,1.4,10.3,926
1076,yuseong1_250224143512_0,36.401399,127.362489,대전광역시 유성구 장동 241-5,2025,2,24,14,0,winter,...,1434,2.82,-9.0,1200,-9.0,-9.00,-9.0,1.4,10.3,926
1077,yuseong1_250224144720_1513,36.372210,127.320740,대전광역시 유성구 지족동 976,2025,2,24,14,0,winter,...,1434,2.82,-9.0,1200,-9.0,-9.00,-9.0,1.4,10.3,926


In [68]:
# DATE 컬럼이 이미 존재한다고 가정
df['DATE'] = pd.to_datetime(df['DATE'])

# --- 요일 정보 ---
# WEEKDAY: 0=월, 6=일 (이미 존재하므로 그대로 사용)
df['is_weekend'] = df['WEEKDAY'].isin([5, 6]).astype(int)

# --- 교통량 기본 값 (평균 300~1200 사이) ---
base_traffic = np.random.randint(300, 1201, size=len(df))

# --- 요일별 보정값 ---
weekday_factor = np.where(df['is_weekend'] == 1, 0.75, 1.15)

# --- 날씨 영향 ---
# RN_DAY: 일강수량, SD_DAY: 일적설량
rain_factor = np.where(df['RN_DAY'] > 0, np.random.uniform(0.80, 0.90, len(df)), 1.0)
snow_factor = np.where(df['SD_DAY'] > 0, np.random.uniform(0.60, 0.80, len(df)), 1.0)

# --- 최종 교통량 ---
df['traffic_volume'] = (
    base_traffic * weekday_factor * rain_factor * snow_factor
).astype(int)

# 비정상 값 제한
df['traffic_volume'] = df['traffic_volume'].clip(120, 2000)


# ===== 점유율(Occupancy) 생성 =====

# 점유율 기본값: 5~35%
base_occp = np.random.uniform(5, 35, size=len(df))

# 주중/주말 영향
weekday_occp_factor = np.where(df['is_weekend'] == 1, 0.85, 1.10)

# 날씨 영향 (비/눈 → 점유율 증가)
weather_occp_factor = np.where(df['RN_DAY'] > 0, 1.10, 1.0)
weather_occp_factor *= np.where(df['SD_DAY'] > 0, 1.15, 1.0)

df['occupancy'] = (
    base_occp * weekday_occp_factor * weather_occp_factor
).round(2)

# 점유율 5~60%로 제한
df['occupancy'] = df['occupancy'].clip(5, 60)

df[['DATE', 'WEEKDAY', 'traffic_volume', 'occupancy']].head()


,DATE,WEEKDAY,traffic_volume,occupancy
0,2025-07-14,0,365,19.45
1,2025-11-06,3,1032,24.48
2,2025-11-06,3,668,11.49
3,2025-11-06,3,964,34.88
4,2025-07-12,5,237,31.46


📊 이 코드가 반영하는 현실적 교통 패턴
✔ 1) 요일 영향

주중: 교통량 증가 (115%)

주말: 교통량 감소 (75%)

✔ 2) 기상 영향

비(RN_DAY > 0): 교통량 -10% ~ -20%, 점유율 +10%

눈(SD_DAY > 0): 교통량 -20% ~ -40%, 점유율 +15%

✔ 3) 기본 범위 유지

교통량: 120 ~ 2000 (비정상적 값 제거)

점유율: 5% ~ 60% (혼잡 로드 기준 최대)

In [69]:
df

,DTCT_UID,PLC_LTTD,PLC_LGTD,ROAD_INFO,YEAR,MONTH,DAY,HOUR,WEEKDAY,SEASON,...,RN_D99,SD_DAY,SD_NEW,SD_MAX,HM_AVG,SI_DAY,SS_DAY,traffic_volume,occupancy,is_weekend
0,gnsoft1_250714084125_11277,36.346661,127.353838,대전광역시 서구 월평동 114-4,2025,7,14,8,0,summer,...,1200,22.9,2.33,18.3,29.4,4.0,3,365,19.45,0
1,gnsoft1_251106093349_24,36.390977,127.407906,대전광역시 유성구 문지동 661-3,2025,11,6,9,3,autumn,...,1200,-9.0,-9.00,-9.0,10.8,6.4,838,1032,24.48,0
2,gnsoft1_251106093431_32,36.390877,127.407925,대전광역시 유성구 문지동 661-3,2025,11,6,9,3,autumn,...,1200,-9.0,-9.00,-9.0,10.8,6.4,838,668,11.49,0
3,gnsoft1_251106194824_90,36.350716,127.368103,대전광역시 서구 갈마동 690,2025,11,6,19,3,autumn,...,1200,-9.0,-9.00,-9.0,10.8,6.4,838,964,34.88,0
4,gnsoft2_250712155250_39377,36.348692,127.335768,대전광역시 유성구 상대동 556,2025,7,12,15,5,summer,...,1200,-9.0,-9.00,-9.0,37.8,13.9,1,237,31.46,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1074,yuseong1_250221140321_0,36.411699,127.377649,대전광역시 유성구 화암동 156-8,2025,2,21,14,4,winter,...,1200,0.4,-9.00,-9.0,1.5,10.1,836,331,31.55,0
1075,yuseong1_250224143428_15517,36.405715,127.368060,대전광역시 유성구 방현동 350,2025,2,24,14,0,winter,...,1200,-9.0,-9.00,-9.0,1.4,10.3,926,631,23.22,0
1076,yuseong1_250224143512_0,36.401399,127.362489,대전광역시 유성구 장동 241-5,2025,2,24,14,0,winter,...,1200,-9.0,-9.00,-9.0,1.4,10.3,926,1088,25.51,0
1077,yuseong1_250224144720_1513,36.372210,127.320740,대전광역시 유성구 지족동 976,2025,2,24,14,0,winter,...,1200,-9.0,-9.00,-9.0,1.4,10.3,926,340,39.03,0


In [70]:
df.to_csv("data_4.csv", index=False, encoding="utf-8-sig")

## 미발생 지역 추가 (임의)

### 기존 방식

In [2]:
df = pd.read_csv("./데이터/data_4.csv")
df

,DTCT_UID,PLC_LTTD,PLC_LGTD,ROAD_INFO,YEAR,MONTH,DAY,HOUR,WEEKDAY,SEASON,...,RN_D99,SD_DAY,SD_NEW,SD_MAX,HM_AVG,SI_DAY,SS_DAY,traffic_volume,occupancy,is_weekend
0,gnsoft1_250714084125_11277,36.346661,127.353838,대전광역시 서구 월평동 114-4,2025,7,14,8,0,summer,...,1200,22.9,2.33,18.3,29.4,4.0,3,365,19.45,0
1,gnsoft1_251106093349_24,36.390977,127.407906,대전광역시 유성구 문지동 661-3,2025,11,6,9,3,autumn,...,1200,-9.0,-9.00,-9.0,10.8,6.4,838,1032,24.48,0
2,gnsoft1_251106093431_32,36.390877,127.407925,대전광역시 유성구 문지동 661-3,2025,11,6,9,3,autumn,...,1200,-9.0,-9.00,-9.0,10.8,6.4,838,668,11.49,0
3,gnsoft1_251106194824_90,36.350716,127.368103,대전광역시 서구 갈마동 690,2025,11,6,19,3,autumn,...,1200,-9.0,-9.00,-9.0,10.8,6.4,838,964,34.88,0
4,gnsoft2_250712155250_39377,36.348692,127.335768,대전광역시 유성구 상대동 556,2025,7,12,15,5,summer,...,1200,-9.0,-9.00,-9.0,37.8,13.9,1,237,31.46,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1074,yuseong1_250221140321_0,36.411699,127.377649,대전광역시 유성구 화암동 156-8,2025,2,21,14,4,winter,...,1200,0.4,-9.00,-9.0,1.5,10.1,836,331,31.55,0
1075,yuseong1_250224143428_15517,36.405715,127.368060,대전광역시 유성구 방현동 350,2025,2,24,14,0,winter,...,1200,-9.0,-9.00,-9.0,1.4,10.3,926,631,23.22,0
1076,yuseong1_250224143512_0,36.401399,127.362489,대전광역시 유성구 장동 241-5,2025,2,24,14,0,winter,...,1200,-9.0,-9.00,-9.0,1.4,10.3,926,1088,25.51,0
1077,yuseong1_250224144720_1513,36.372210,127.320740,대전광역시 유성구 지족동 976,2025,2,24,14,0,winter,...,1200,-9.0,-9.00,-9.0,1.4,10.3,926,340,39.03,0


In [ ]:
# 생성 수
N = 3000

# 1) 발생 데이터에서 N개 샘플링
sampled = df.sample(N, replace=True).copy()

# 2) UID 변경
sampled['DTCT_UID'] = ["NON_" + str(i) for i in range(N)]

# 3) 좌표만 주변에 랜덤 offset 적용
lat_offset = np.random.uniform(-0.0008, 0.0008, size=N)
lon_offset = np.random.uniform(-0.0008, 0.0008, size=N)

sampled['PLC_LTTD'] = sampled['PLC_LTTD'] + lat_offset
sampled['PLC_LGTD'] = sampled['PLC_LGTD'] + lon_offset

# 4) 포트홀/균열 관련만 0으로 설정
sampled['type_pothole']   = 0
sampled['type_crack']     = 0
sampled['crack_rate']     = 0
sampled['complex_damage'] = 0

# 5) 최종 데이터 결합
df_full = pd.concat([df, sampled], ignore_index=True)


In [85]:
df_full['type_pothole'].value_counts()

type_pothole
0    3000
1    1079
Name: count, dtype: int64

In [86]:
df_full

,DTCT_UID,PLC_LTTD,PLC_LGTD,ROAD_INFO,YEAR,MONTH,DAY,HOUR,WEEKDAY,SEASON,...,RN_D99,SD_DAY,SD_NEW,SD_MAX,HM_AVG,SI_DAY,SS_DAY,traffic_volume,occupancy,is_weekend
0,gnsoft1_250714084125_11277,36.346661,127.353838,대전광역시 서구 월평동 114-4,2025,7,14,8,0,summer,...,1200,22.9,2.33,18.3,29.4,4.0,3,365,19.45,0
1,gnsoft1_251106093349_24,36.390977,127.407906,대전광역시 유성구 문지동 661-3,2025,11,6,9,3,autumn,...,1200,-9.0,-9.00,-9.0,10.8,6.4,838,1032,24.48,0
2,gnsoft1_251106093431_32,36.390877,127.407925,대전광역시 유성구 문지동 661-3,2025,11,6,9,3,autumn,...,1200,-9.0,-9.00,-9.0,10.8,6.4,838,668,11.49,0
3,gnsoft1_251106194824_90,36.350716,127.368103,대전광역시 서구 갈마동 690,2025,11,6,19,3,autumn,...,1200,-9.0,-9.00,-9.0,10.8,6.4,838,964,34.88,0
4,gnsoft2_250712155250_39377,36.348692,127.335768,대전광역시 유성구 상대동 556,2025,7,12,15,5,summer,...,1200,-9.0,-9.00,-9.0,37.8,13.9,1,237,31.46,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4074,NON_2995,36.356212,127.342788,대전광역시 유성구 봉명동 574,2024,3,23,8,5,spring,...,1200,-9.0,2.25,-9.0,15.9,9.3,2254,714,16.67,1
4075,NON_2996,36.398007,127.314908,대전광역시 유성구 외삼동 577,2024,3,20,9,2,spring,...,1100,-9.0,-9.00,-9.0,8.7,9.9,947,1046,16.36,0
4076,NON_2997,36.419990,127.379449,대전광역시 유성구 덕진동 산 14-3,2024,2,29,12,3,winter,...,1400,0.2,3.00,-9.0,5.5,0.0,49,553,12.93,0
4077,NON_2998,36.368507,127.318042,대전광역시 유성구 노은동 583,2024,4,6,8,5,spring,...,1200,-9.0,-9.00,-9.0,14.0,5.3,808,485,15.54,1


In [87]:
df_full.to_csv("data_5.csv", index=False, encoding="utf-8-sig")

### 새로운 방식

In [17]:
# 1) 데이터 로드
df = pd.read_csv("./데이터/data_4.csv")

# 2) synthetic 미발생 데이터 개수
N_SYN = 18000   # 원하는 만큼 조절 가능

rng = np.random.default_rng(42)

# 3) 원본에서 파손 발생 데이터만 seed로 사용
df_seed = df.copy()

# 4) 필요 컬럼 복사 기반으로 생성
synthetic = df_seed.sample(n=N_SYN, replace=True, random_state=42).reset_index(drop=True)

# 5) UID 새로 생성
synthetic["DTCT_UID"] = [
    f"SYN_{i:06d}" for i in range(N_SYN)
]

# 6) 좌표를 주변 랜덤 값으로 이동
#    약 ± 0.0005 → 약 50m ~ 70m 수준
synthetic["PLC_LTTD"] = synthetic["PLC_LTTD"] + rng.uniform(-0.0005, 0.0005, N_SYN)
synthetic["PLC_LGTD"] = synthetic["PLC_LGTD"] + rng.uniform(-0.0005, 0.0005, N_SYN)

# 7) 포트홀 미발생 데이터이므로 type_pothole = 0
synthetic["type_pothole"] = 0

# 8) 균열율 생성 (현실 분포)
crack_dist = rng.choice(
    [0, 1, 2],
    size=N_SYN,
    p=[0.7, 0.2, 0.1]
)

crack_rates = []
for c in crack_dist:
    if c == 0:
        crack_rates.append(rng.uniform(0, 5))
    elif c == 1:
        crack_rates.append(rng.uniform(5, 15))
    else:
        crack_rates.append(rng.uniform(15, 30))

synthetic["crack_rate"] = crack_rates

# 9) 최종 병합
df_final = pd.concat([df, synthetic], ignore_index=True)

print("최종 데이터 크기:", len(df_final))
print("균열율 분포 요약:\n", synthetic["crack_rate"].describe())

최종 데이터 크기: 19079
균열율 분포 요약:
 count    18000.000000
mean         6.012053
std          6.590379
min          0.000066
25%          1.829643
50%          3.604418
75%          7.557029
max         29.994772
Name: crack_rate, dtype: float64


In [19]:
df_final

,DTCT_UID,PLC_LTTD,PLC_LGTD,ROAD_INFO,YEAR,MONTH,DAY,HOUR,WEEKDAY,SEASON,...,RN_D99,SD_DAY,SD_NEW,SD_MAX,HM_AVG,SI_DAY,SS_DAY,traffic_volume,occupancy,is_weekend
0,gnsoft1_250714084125_11277,36.346661,127.353838,대전광역시 서구 월평동 114-4,2025,7,14,8,0,summer,...,1200,22.9,2.33,18.3,29.4,4.0,3,365,19.45,0
1,gnsoft1_251106093349_24,36.390977,127.407906,대전광역시 유성구 문지동 661-3,2025,11,6,9,3,autumn,...,1200,-9.0,-9.00,-9.0,10.8,6.4,838,1032,24.48,0
2,gnsoft1_251106093431_32,36.390877,127.407925,대전광역시 유성구 문지동 661-3,2025,11,6,9,3,autumn,...,1200,-9.0,-9.00,-9.0,10.8,6.4,838,668,11.49,0
3,gnsoft1_251106194824_90,36.350716,127.368103,대전광역시 서구 갈마동 690,2025,11,6,19,3,autumn,...,1200,-9.0,-9.00,-9.0,10.8,6.4,838,964,34.88,0
4,gnsoft2_250712155250_39377,36.348692,127.335768,대전광역시 유성구 상대동 556,2025,7,12,15,5,summer,...,1200,-9.0,-9.00,-9.0,37.8,13.9,1,237,31.46,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19074,SYN_017995,36.383874,127.364114,대전광역시 유성구 가정동 산 16-5,2024,2,28,12,2,winter,...,1200,-9.0,-9.00,-9.0,5.9,3.7,916,852,22.85,0
19075,SYN_017996,36.361130,127.345506,대전광역시 유성구 궁동 485,2024,3,25,13,0,spring,...,1000,26.1,11.25,-9.0,10.0,0.0,12,525,16.58,0
19076,SYN_017997,36.424535,127.293709,세종특별자치시 금남면 두만리 52-8,2024,3,15,14,4,spring,...,1200,-9.0,-9.00,-9.0,10.5,11.0,2301,848,18.58,0
19077,SYN_017998,36.348990,127.323735,대전광역시 유성구 구암동 221-4,2024,3,27,10,2,spring,...,1200,1.2,-9.00,-9.0,11.6,9.0,711,713,11.31,0


In [20]:
df_final.to_csv("./데이터/data_6.csv", index=False, encoding="utf-8-sig")

## 보수이력 추가 (임의)

### 기존방식

In [88]:
df = pd.read_csv("data_5.csv")
df

,DTCT_UID,PLC_LTTD,PLC_LGTD,ROAD_INFO,YEAR,MONTH,DAY,HOUR,WEEKDAY,SEASON,...,RN_D99,SD_DAY,SD_NEW,SD_MAX,HM_AVG,SI_DAY,SS_DAY,traffic_volume,occupancy,is_weekend
0,gnsoft1_250714084125_11277,36.346661,127.353838,대전광역시 서구 월평동 114-4,2025,7,14,8,0,summer,...,1200,22.9,2.33,18.3,29.4,4.0,3,365,19.45,0
1,gnsoft1_251106093349_24,36.390977,127.407906,대전광역시 유성구 문지동 661-3,2025,11,6,9,3,autumn,...,1200,-9.0,-9.00,-9.0,10.8,6.4,838,1032,24.48,0
2,gnsoft1_251106093431_32,36.390877,127.407925,대전광역시 유성구 문지동 661-3,2025,11,6,9,3,autumn,...,1200,-9.0,-9.00,-9.0,10.8,6.4,838,668,11.49,0
3,gnsoft1_251106194824_90,36.350716,127.368103,대전광역시 서구 갈마동 690,2025,11,6,19,3,autumn,...,1200,-9.0,-9.00,-9.0,10.8,6.4,838,964,34.88,0
4,gnsoft2_250712155250_39377,36.348692,127.335768,대전광역시 유성구 상대동 556,2025,7,12,15,5,summer,...,1200,-9.0,-9.00,-9.0,37.8,13.9,1,237,31.46,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4074,NON_2995,36.356212,127.342788,대전광역시 유성구 봉명동 574,2024,3,23,8,5,spring,...,1200,-9.0,2.25,-9.0,15.9,9.3,2254,714,16.67,1
4075,NON_2996,36.398007,127.314908,대전광역시 유성구 외삼동 577,2024,3,20,9,2,spring,...,1100,-9.0,-9.00,-9.0,8.7,9.9,947,1046,16.36,0
4076,NON_2997,36.419990,127.379449,대전광역시 유성구 덕진동 산 14-3,2024,2,29,12,3,winter,...,1400,0.2,3.00,-9.0,5.5,0.0,49,553,12.93,0
4077,NON_2998,36.368507,127.318042,대전광역시 유성구 노은동 583,2024,4,6,8,5,spring,...,1200,-9.0,-9.00,-9.0,14.0,5.3,808,485,15.54,1


In [89]:
df.columns

Index(['DTCT_UID', 'PLC_LTTD', 'PLC_LGTD', 'ROAD_INFO', 'YEAR', 'MONTH', 'DAY',
       'HOUR', 'WEEKDAY', 'SEASON', 'COMMUTE', 'COUNT_7D', 'COUNT_30D',
       'COUNT_90D', 'crack_rate', 'DATE', 'type_pothole', 'type_crack',
       'complex_damage', 'TA_AVG', 'TA_MAX', 'TA_MIN', 'RN_DAY', 'RN_DUR',
       'RN_D99', 'SD_DAY', 'SD_NEW', 'SD_MAX', 'HM_AVG', 'SI_DAY', 'SS_DAY',
       'traffic_volume', 'occupancy', 'is_weekend'],
      dtype='object')

In [91]:
pothole_df = df[df['type_pothole'] == 1].copy()

# 1) 기본 조건 파생
pothole_df['high_crack'] = (pothole_df['crack_rate'] >= 10).astype(int)
pothole_df['mid_crack']  = (pothole_df['crack_rate'].between(5, 10)).astype(int)
pothole_df['low_crack']  = (pothole_df['crack_rate'] < 5).astype(int)

pothole_df['repeat_damage'] = (pothole_df['COUNT_30D'] >= 5).astype(int)
pothole_df['light_repeat']  = (pothole_df['COUNT_30D'].between(2, 4)).astype(int)

pothole_df['heavy_rain_day'] = (pothole_df['RN_DAY'] > 20).astype(int)

# 2) 공법 선택 로직
def choose_method(row):
    if row['high_crack'] and row['complex_damage'] or row['repeat_damage']:
        return "Full_Depth"
    if row['mid_crack'] or row['light_repeat']:
        return "Mill_Overlay"
    if row['type_crack'] == 1:
        return "Crack_Seal"
    return "Patch"

pothole_df['REPAIR_TYPE'] = pothole_df.apply(choose_method, axis=1)

# 3) 공법별 보수 확률
def repair_probability(rt):
    if rt == "Full_Depth": return 0.80
    if rt == "Mill_Overlay": return 0.60
    if rt == "Crack_Seal": return 0.50
    if rt == "Patch": return 0.40
    return 0.10

pothole_df['REPAIR_PROB'] = pothole_df['REPAIR_TYPE'].apply(repair_probability)

# 4) 실제 보수 발생 여부 결정
pothole_df['HAS_REPAIR'] = (np.random.rand(len(pothole_df)) < pothole_df['REPAIR_PROB']).astype(int)

repair_df = pothole_df[pothole_df['HAS_REPAIR'] == 1].copy()

# 5) 공법별 보수 날짜 생성
def generate_repair_date(row):
    base_date = pd.to_datetime(row['DATE'])

    if row['REPAIR_TYPE'] == "Full_Depth":
        days = np.random.randint(14, 60)
    elif row['REPAIR_TYPE'] == "Mill_Overlay":
        days = np.random.randint(30, 120)
    elif row['REPAIR_TYPE'] == "Patch":
        days = np.random.randint(7, 60)
    elif row['REPAIR_TYPE'] == "Crack_Seal":
        days = np.random.randint(14, 90)
    else:
        days = np.random.randint(30, 180)

    return base_date + pd.to_timedelta(days, unit="D")

repair_df['REPAIR_DATE'] = repair_df.apply(generate_repair_date, axis=1)

# 6) 필요한 컬럼만 이력으로 남기기
repair_df = repair_df[[
    "DTCT_UID", "ROAD_INFO", "REPAIR_TYPE", "REPAIR_DATE",
    "PLC_LTTD", "PLC_LGTD"
]]

repair_df = repair_df.rename(columns={
    "PLC_LTTD": "REPAIR_LAT",
    "PLC_LGTD": "REPAIR_LON"
})

# 7) df_full에 merge
df_full = df_full.merge(
    repair_df[["DTCT_UID", "REPAIR_TYPE", "REPAIR_DATE", "REPAIR_LAT", "REPAIR_LON"]],
    on="DTCT_UID", how="left"
)

df_full['HAS_REPAIR'] = df_full['REPAIR_DATE'].notnull().astype(int)

df_full['DAYS_TO_REPAIR'] = (
    pd.to_datetime(df_full['REPAIR_DATE']) - pd.to_datetime(df_full['DATE'])
).dt.days


| 컬럼명           | 설명                                                                 |
| ------------- | ------------------------------------------------------------------ |
| `DTCT_UID`    | 보수의 기준이 된 파손 이벤트 ID (포트홀 발생 건 중 실제 보수가 일어난 것만 포함)                  |
| `ROAD_INFO`   | 보수가 수행된 도로 구간 ID                                                   |
| `REPAIR_TYPE` | 선택된 보수 공법 종류 (`Patch`, `Mill_Overlay`, `Full_Depth`, `Crack_Seal`) |
| `REPAIR_DATE` | 해당 파손에 대해 실제로 보수가 이루어진 날짜 (발생일 + 공법별 랜덤 경과일)                       |
| `REPAIR_LAT`  | 보수 위치 위도 (기준 파손의 `PLC_LTTD` 값 사용)                                  |
| `REPAIR_LON`  | 보수 위치 경도 (기준 파손의 `PLC_LGTD` 값 사용)                                  |

| 컬럼명              | 설명                                                                                             |
| ---------------- | ---------------------------------------------------------------------------------------------- |
| `REPAIR_TYPE`    | 해당 파손 이벤트에 대해 선택된 보수 공법 (`Patch`, `Mill_Overlay`, `Full_Depth`, `Crack_Seal`)· 보수가 없는 행은 `NaN` |
| `REPAIR_DATE`    | 보수 수행 날짜 · 보수가 없는 행은 `NaN`                                                                     |
| `REPAIR_LAT`     | 보수 위치 위도 · 보수가 없는 행은 `NaN`                                                                     |
| `REPAIR_LON`     | 보수 위치 경도 · 보수가 없는 행은 `NaN`                                                                     |
| `HAS_REPAIR`     | 이 파손 건에 대해 실제 보수가 있었는지 여부 (있으면 `1`, 없으면 `0`)                                                   |
| `DAYS_TO_REPAIR` | 파손 발생일(`DATE`)로부터 보수일까지의 경과일수 · 보수 없는 행은 `NaN`                                                 |


In [95]:
df_full['REPAIR_TYPE'].value_counts()

REPAIR_TYPE
Mill_Overlay    210
Patch           179
Full_Depth      151
Crack_Seal       27
Name: count, dtype: int64

In [96]:
df_full

,DTCT_UID,PLC_LTTD,PLC_LGTD,ROAD_INFO,YEAR,MONTH,DAY,HOUR,WEEKDAY,SEASON,...,SS_DAY,traffic_volume,occupancy,is_weekend,REPAIR_TYPE,REPAIR_DATE,REPAIR_LAT,REPAIR_LON,HAS_REPAIR,DAYS_TO_REPAIR
0,gnsoft1_250714084125_11277,36.346661,127.353838,대전광역시 서구 월평동 114-4,2025,7,14,8,0,summer,...,3,365,19.45,0,NaN,NaT,NaN,NaN,0,NaN
1,gnsoft1_251106093349_24,36.390977,127.407906,대전광역시 유성구 문지동 661-3,2025,11,6,9,3,autumn,...,838,1032,24.48,0,NaN,NaT,NaN,NaN,0,NaN
2,gnsoft1_251106093431_32,36.390877,127.407925,대전광역시 유성구 문지동 661-3,2025,11,6,9,3,autumn,...,838,668,11.49,0,NaN,NaT,NaN,NaN,0,NaN
3,gnsoft1_251106194824_90,36.350716,127.368103,대전광역시 서구 갈마동 690,2025,11,6,19,3,autumn,...,838,964,34.88,0,NaN,NaT,NaN,NaN,0,NaN
4,gnsoft2_250712155250_39377,36.348692,127.335768,대전광역시 유성구 상대동 556,2025,7,12,15,5,summer,...,1,237,31.46,1,NaN,NaT,NaN,NaN,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4074,NON_2995,36.356212,127.342788,대전광역시 유성구 봉명동 574,2024,3,23,8,5,spring,...,2254,714,16.67,1,NaN,NaT,NaN,NaN,0,NaN
4075,NON_2996,36.398007,127.314908,대전광역시 유성구 외삼동 577,2024,3,20,9,2,spring,...,947,1046,16.36,0,NaN,NaT,NaN,NaN,0,NaN
4076,NON_2997,36.419990,127.379449,대전광역시 유성구 덕진동 산 14-3,2024,2,29,12,3,winter,...,49,553,12.93,0,NaN,NaT,NaN,NaN,0,NaN
4077,NON_2998,36.368507,127.318042,대전광역시 유성구 노은동 583,2024,4,6,8,5,spring,...,808,485,15.54,1,NaN,NaT,NaN,NaN,0,NaN


### 새로운 방식

In [22]:
df = pd.read_csv("./데이터/data_6.csv")
df

,DTCT_UID,PLC_LTTD,PLC_LGTD,ROAD_INFO,YEAR,MONTH,DAY,HOUR,WEEKDAY,SEASON,...,RN_D99,SD_DAY,SD_NEW,SD_MAX,HM_AVG,SI_DAY,SS_DAY,traffic_volume,occupancy,is_weekend
0,gnsoft1_250714084125_11277,36.346661,127.353838,대전광역시 서구 월평동 114-4,2025,7,14,8,0,summer,...,1200,22.9,2.33,18.3,29.4,4.0,3,365,19.45,0
1,gnsoft1_251106093349_24,36.390977,127.407906,대전광역시 유성구 문지동 661-3,2025,11,6,9,3,autumn,...,1200,-9.0,-9.00,-9.0,10.8,6.4,838,1032,24.48,0
2,gnsoft1_251106093431_32,36.390877,127.407925,대전광역시 유성구 문지동 661-3,2025,11,6,9,3,autumn,...,1200,-9.0,-9.00,-9.0,10.8,6.4,838,668,11.49,0
3,gnsoft1_251106194824_90,36.350716,127.368103,대전광역시 서구 갈마동 690,2025,11,6,19,3,autumn,...,1200,-9.0,-9.00,-9.0,10.8,6.4,838,964,34.88,0
4,gnsoft2_250712155250_39377,36.348692,127.335768,대전광역시 유성구 상대동 556,2025,7,12,15,5,summer,...,1200,-9.0,-9.00,-9.0,37.8,13.9,1,237,31.46,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19074,SYN_017995,36.383874,127.364114,대전광역시 유성구 가정동 산 16-5,2024,2,28,12,2,winter,...,1200,-9.0,-9.00,-9.0,5.9,3.7,916,852,22.85,0
19075,SYN_017996,36.361130,127.345506,대전광역시 유성구 궁동 485,2024,3,25,13,0,spring,...,1000,26.1,11.25,-9.0,10.0,0.0,12,525,16.58,0
19076,SYN_017997,36.424535,127.293709,세종특별자치시 금남면 두만리 52-8,2024,3,15,14,4,spring,...,1200,-9.0,-9.00,-9.0,10.5,11.0,2301,848,18.58,0
19077,SYN_017998,36.348990,127.323735,대전광역시 유성구 구암동 221-4,2024,3,27,10,2,spring,...,1200,1.2,-9.00,-9.0,11.6,9.0,711,713,11.31,0


| 컬럼명                | 설명             |
| ------------------ | -------------- |
| **HAS_REPAIR**     | 보수 시행 여부 (1/0) |
| **REPAIR_DATE**    | 보수 날짜          |
| **REPAIR_LAT**     | 보수 위치 위도       |
| **REPAIR_LON**     | 보수 위치 경도       |
| **REPAIR_TYPE**    | 보수 공법          |
| **DAYS_TO_REPAIR** | 파손→보수까지 걸린 일수  |

In [23]:
df['DATE'] = pd.to_datetime(df['DATE'])

rng = np.random.default_rng(42)

CUTOFF_DATE = pd.Timestamp("2025-11-26")   # 미래 날짜 방지
NO_REPAIR_AFTER = pd.Timestamp("2025-09-01")  # 최근 9월 이후 건은 보수 없음

# 2) 보수 확률 규칙
def repair_probability(row):
    # 최근 데이터는 무조건 보수 없음
    if row['DATE'] >= NO_REPAIR_AFTER:
        return 0.0
    
    # 위험 구간
    if row["crack_rate"] >= 15 or row.get("type_pothole", 0) == 1:
        return 0.6

    # 보통 구간
    elif row["crack_rate"] >= 5:
        return 0.25

    # 좋음 구간
    else:
        return 0.05

df["repair_prob"] = df.apply(repair_probability, axis=1)

# 3) HAS_REPAIR 생성
df["HAS_REPAIR"] = (rng.uniform(0, 1, len(df)) < df["repair_prob"]).astype(int)

# 최근 건은 무조건 보수 없음
df.loc[df["DATE"] >= NO_REPAIR_AFTER, "HAS_REPAIR"] = 0


# 4) 보수 날짜 생성
repair_delay = rng.integers(5, 60, size=len(df))
df["REPAIR_DATE"] = df["DATE"] + pd.to_timedelta(repair_delay, unit="D")

# 미래 날짜 방지
df.loc[df["REPAIR_DATE"] > CUTOFF_DATE, "REPAIR_DATE"] = CUTOFF_DATE

# 최근 건 → 보수 없음 → 날짜 제거
df.loc[df["HAS_REPAIR"] == 0, "REPAIR_DATE"] = pd.NaT


# 5) 보수 공법 생성
repair_methods = ["CrackSealing", "Patch", "Overlay", "FullDepth"]

df["REPAIR_TYPE"] = None
df.loc[df["HAS_REPAIR"] == 1, "REPAIR_TYPE"] = rng.choice(
    repair_methods, 
    size=df["HAS_REPAIR"].sum(),
    p=[0.4, 0.3, 0.2, 0.1]
)


# 6) 보수 좌표 생성 (발생 위치 ± 30~50m)
df["REPAIR_LAT"] = df["PLC_LTTD"]
df["REPAIR_LON"] = df["PLC_LGTD"]

offset_lat = rng.uniform(-0.0003, 0.0003, size=len(df))
offset_lon = rng.uniform(-0.0003, 0.0003, size=len(df))

df.loc[df["HAS_REPAIR"] == 1, "REPAIR_LAT"] += offset_lat[df["HAS_REPAIR"] == 1]
df.loc[df["HAS_REPAIR"] == 1, "REPAIR_LON"] += offset_lon[df["HAS_REPAIR"] == 1]

# HAS_REPAIR=0 → Null
df.loc[df["HAS_REPAIR"] == 0, ["REPAIR_LAT", "REPAIR_LON"]] = np.nan


# 7) 경과일(DAYS_TO_REPAIR) 생성
df["DAYS_TO_REPAIR"] = (df["REPAIR_DATE"] - df["DATE"]).dt.days
df.loc[df["HAS_REPAIR"] == 0, "DAYS_TO_REPAIR"] = np.nan


# 8) 정리 & 최종 출력
df.drop(columns=["repair_prob"], inplace=True)

print("최종 데이터 크기:", len(df))
print(df[["HAS_REPAIR", "REPAIR_TYPE", "REPAIR_DATE"]].head())

# 저장 (선택)
# df.to_csv("./데이터/data_7_with_repair.csv", index=False, encoding="utf-8-sig")

최종 데이터 크기: 19079
   HAS_REPAIR   REPAIR_TYPE REPAIR_DATE
0           0          None         NaT
1           0          None         NaT
2           0          None         NaT
3           0          None         NaT
4           1  CrackSealing  2025-08-10


In [25]:
df

,DTCT_UID,PLC_LTTD,PLC_LGTD,ROAD_INFO,YEAR,MONTH,DAY,HOUR,WEEKDAY,SEASON,...,SS_DAY,traffic_volume,occupancy,is_weekend,HAS_REPAIR,REPAIR_DATE,REPAIR_TYPE,REPAIR_LAT,REPAIR_LON,DAYS_TO_REPAIR
0,gnsoft1_250714084125_11277,36.346661,127.353838,대전광역시 서구 월평동 114-4,2025,7,14,8,0,summer,...,3,365,19.45,0,0,NaT,None,NaN,NaN,NaN
1,gnsoft1_251106093349_24,36.390977,127.407906,대전광역시 유성구 문지동 661-3,2025,11,6,9,3,autumn,...,838,1032,24.48,0,0,NaT,None,NaN,NaN,NaN
2,gnsoft1_251106093431_32,36.390877,127.407925,대전광역시 유성구 문지동 661-3,2025,11,6,9,3,autumn,...,838,668,11.49,0,0,NaT,None,NaN,NaN,NaN
3,gnsoft1_251106194824_90,36.350716,127.368103,대전광역시 서구 갈마동 690,2025,11,6,19,3,autumn,...,838,964,34.88,0,0,NaT,None,NaN,NaN,NaN
4,gnsoft2_250712155250_39377,36.348692,127.335768,대전광역시 유성구 상대동 556,2025,7,12,15,5,summer,...,1,237,31.46,1,1,2025-08-10,CrackSealing,36.348714,127.335513,29.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19074,SYN_017995,36.383874,127.364114,대전광역시 유성구 가정동 산 16-5,2024,2,28,12,2,winter,...,916,852,22.85,0,0,NaT,None,NaN,NaN,NaN
19075,SYN_017996,36.361130,127.345506,대전광역시 유성구 궁동 485,2024,3,25,13,0,spring,...,12,525,16.58,0,1,2024-04-09,CrackSealing,36.361030,127.345799,15.0
19076,SYN_017997,36.424535,127.293709,세종특별자치시 금남면 두만리 52-8,2024,3,15,14,4,spring,...,2301,848,18.58,0,0,NaT,None,NaN,NaN,NaN
19077,SYN_017998,36.348990,127.323735,대전광역시 유성구 구암동 221-4,2024,3,27,10,2,spring,...,711,713,11.31,0,0,NaT,None,NaN,NaN,NaN


In [26]:
df['HAS_REPAIR'].value_counts()

HAS_REPAIR
0    16503
1     2576
Name: count, dtype: int64

In [27]:
df.columns

Index(['DTCT_UID', 'PLC_LTTD', 'PLC_LGTD', 'ROAD_INFO', 'YEAR', 'MONTH', 'DAY',
       'HOUR', 'WEEKDAY', 'SEASON', 'COMMUTE', 'COUNT_7D', 'COUNT_30D',
       'COUNT_90D', 'crack_rate', 'DATE', 'type_pothole', 'type_crack',
       'complex_damage', 'TA_AVG', 'TA_MAX', 'TA_MIN', 'RN_DAY', 'RN_DUR',
       'RN_D99', 'SD_DAY', 'SD_NEW', 'SD_MAX', 'HM_AVG', 'SI_DAY', 'SS_DAY',
       'traffic_volume', 'occupancy', 'is_weekend', 'HAS_REPAIR',
       'REPAIR_DATE', 'REPAIR_TYPE', 'REPAIR_LAT', 'REPAIR_LON',
       'DAYS_TO_REPAIR'],
      dtype='object')

In [28]:
df.to_csv("./데이터/data_final.csv", index=False, encoding="utf-8-sig")

## 라벨 생성

In [3]:
df = pd.read_csv("./데이터/data_final.csv")
df

,DTCT_UID,PLC_LTTD,PLC_LGTD,ROAD_INFO,YEAR,MONTH,DAY,HOUR,WEEKDAY,SEASON,...,SS_DAY,traffic_volume,occupancy,is_weekend,HAS_REPAIR,REPAIR_DATE,REPAIR_TYPE,REPAIR_LAT,REPAIR_LON,DAYS_TO_REPAIR
0,gnsoft1_250714084125_11277,36.346661,127.353838,대전광역시 서구 월평동 114-4,2025,7,14,8,0,summer,...,3,365,19.45,0,0,NaN,NaN,NaN,NaN,NaN
1,gnsoft1_251106093349_24,36.390977,127.407906,대전광역시 유성구 문지동 661-3,2025,11,6,9,3,autumn,...,838,1032,24.48,0,0,NaN,NaN,NaN,NaN,NaN
2,gnsoft1_251106093431_32,36.390877,127.407925,대전광역시 유성구 문지동 661-3,2025,11,6,9,3,autumn,...,838,668,11.49,0,0,NaN,NaN,NaN,NaN,NaN
3,gnsoft1_251106194824_90,36.350716,127.368103,대전광역시 서구 갈마동 690,2025,11,6,19,3,autumn,...,838,964,34.88,0,0,NaN,NaN,NaN,NaN,NaN
4,gnsoft2_250712155250_39377,36.348692,127.335768,대전광역시 유성구 상대동 556,2025,7,12,15,5,summer,...,1,237,31.46,1,1,2025-08-10,CrackSealing,36.348714,127.335513,29.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19074,SYN_017995,36.383874,127.364114,대전광역시 유성구 가정동 산 16-5,2024,2,28,12,2,winter,...,916,852,22.85,0,0,NaN,NaN,NaN,NaN,NaN
19075,SYN_017996,36.361130,127.345506,대전광역시 유성구 궁동 485,2024,3,25,13,0,spring,...,12,525,16.58,0,1,2024-04-09,CrackSealing,36.361030,127.345799,15.0
19076,SYN_017997,36.424535,127.293709,세종특별자치시 금남면 두만리 52-8,2024,3,15,14,4,spring,...,2301,848,18.58,0,0,NaN,NaN,NaN,NaN,NaN
19077,SYN_017998,36.348990,127.323735,대전광역시 유성구 구암동 221-4,2024,3,27,10,2,spring,...,711,713,11.31,0,0,NaN,NaN,NaN,NaN,NaN


# 최종 데이터셋

## 데이터 로드

In [2]:
df = pd.read_csv('data_daejeon.csv')
df

,MTDT_INDEX,DTCT_UID,ROAD_POTHOLE,ROAD_CRACK,PLC_LTTD,PLC_LGTD,ROAD_INFO,DSTA_LGLD_INFO,DSTA_ADSTRD_INFO,IMG_CONF
0,2603,RAD01_240119162357_441,1,0,36.384627,127.343459,대전광역시 유성구 신성동 산 26-12,신성동,신성동,NaN
1,2613,RAD01_240119162610_0,1,0,36.398713,127.357864,대전광역시 유성구 장동 202-11,장동,신성동,NaN
2,2614,RAD01_240119162611_0,1,0,36.398783,127.358005,대전광역시 유성구 장동 202-11,장동,신성동,NaN
3,2655,RAD01_240119163618_0,1,0,36.427899,127.387323,대전광역시 유성구 관평동 1286,관평동,관평동,NaN
4,2659,RAD01_240119163629_608,1,0,36.427741,127.387724,대전광역시 유성구 관평동 1286,관평동,관평동,NaN
...,...,...,...,...,...,...,...,...,...,...
1489,23118,gnsoft1_251204085809_886916,1,0,36.390080,127.406226,대전광역시 유성구 문지동 667,문지동,전민동,0.8267
1490,23119,gnsoft1_251204085823_887180,1,0,36.390470,127.406918,대전광역시 유성구 문지동 668,문지동,전민동,0.8347
1491,23120,gnsoft1_251204085855_887652,1,0,36.391149,127.407477,대전광역시 유성구 문지동 676,문지동,전민동,0.8169
1492,23121,gnsoft1_251204085913_887841,1,0,36.390990,127.407868,대전광역시 유성구 문지동 661-3,문지동,전민동,0.8131


## 날짜 정보 추출

In [4]:
# UID에서 날짜(YYMMDD)만 추출하는 함수
def extract_date(uid):
    # UID 형식: prefix_YYMMDDHHMMSS_xxx
    parts = uid.split("_")
    if len(parts) < 3:
        return None
    datetime_part = parts[1]  # YYMMDDHHMMSS
    date_part = datetime_part[:6]  # YYMMDD
    return date_part

# 날짜 컬럼 생성
df["DATE"] = df["DTCT_UID"].apply(extract_date)

# YYMMDD → YEAR, MONTH, DAY 변환
df["YEAR"] = df["DATE"].str[:2].astype(int) + 2000
df["MONTH"] = df["DATE"].str[2:4].astype(int)
df["DAY"] = df["DATE"].str[4:6].astype(int)

df

,MTDT_INDEX,DTCT_UID,ROAD_POTHOLE,ROAD_CRACK,PLC_LTTD,PLC_LGTD,ROAD_INFO,DSTA_LGLD_INFO,DSTA_ADSTRD_INFO,IMG_CONF,DATE,YEAR,MONTH,DAY
0,2603,RAD01_240119162357_441,1,0,36.384627,127.343459,대전광역시 유성구 신성동 산 26-12,신성동,신성동,NaN,240119,2024,1,19
1,2613,RAD01_240119162610_0,1,0,36.398713,127.357864,대전광역시 유성구 장동 202-11,장동,신성동,NaN,240119,2024,1,19
2,2614,RAD01_240119162611_0,1,0,36.398783,127.358005,대전광역시 유성구 장동 202-11,장동,신성동,NaN,240119,2024,1,19
3,2655,RAD01_240119163618_0,1,0,36.427899,127.387323,대전광역시 유성구 관평동 1286,관평동,관평동,NaN,240119,2024,1,19
4,2659,RAD01_240119163629_608,1,0,36.427741,127.387724,대전광역시 유성구 관평동 1286,관평동,관평동,NaN,240119,2024,1,19
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1489,23118,gnsoft1_251204085809_886916,1,0,36.390080,127.406226,대전광역시 유성구 문지동 667,문지동,전민동,0.8267,251204,2025,12,4
1490,23119,gnsoft1_251204085823_887180,1,0,36.390470,127.406918,대전광역시 유성구 문지동 668,문지동,전민동,0.8347,251204,2025,12,4
1491,23120,gnsoft1_251204085855_887652,1,0,36.391149,127.407477,대전광역시 유성구 문지동 676,문지동,전민동,0.8169,251204,2025,12,4
1492,23121,gnsoft1_251204085913_887841,1,0,36.390990,127.407868,대전광역시 유성구 문지동 661-3,문지동,전민동,0.8131,251204,2025,12,4


## 기상청 데이터 통합

In [5]:
import pandas as pd
import requests

authKey = "_FW6tOhdRS2VurToXQUtMw"
stn = 133 #대전

# 1) 날짜 범위 추출
df['DATE'] = pd.to_datetime(df['DATE'])
start_date = df['DATE'].min().strftime("%Y%m%d")
end_date   = df['DATE'].max().strftime("%Y%m%d")

print("조회 범위:", start_date, "~", end_date)

# 2) 기상청 기간조회 API 요청
url = "https://apihub.kma.go.kr/api/typ01/url/kma_sfcdd3.php"

params = {
    "tm1": start_date,
    "tm2": end_date,
    "stn": stn,
    "help": 0,
    "authKey": authKey
}

resp = requests.get(url, params=params)

# 인코딩 자동 처리
resp.encoding = resp.apparent_encoding
lines = resp.text.strip().split("\n")

# 3) 주석 제거 + 공백 split
data_lines = [line for line in lines if not line.startswith('#') and line.strip() != ""]
rows = [line.split() for line in data_lines]

df_weather_raw = pd.DataFrame(rows)

# 4) 날짜 컬럼 변환
df_weather_raw['DATE'] = pd.to_datetime(df_weather_raw[0], format="%Y%m%d", errors='coerce')

# 5) 필요한 컬럼 index 매핑
colmap = {
    "TA_AVG": 9,
    "TA_MAX": 10,
    "TA_MIN": 12,
    "RN_DAY": 36,
    "RN_DUR": 38,
    "RN_D99": 37,
    "SD_DAY": 39,
    "SD_NEW": 40,
    "SD_MAX": 41,
    "HM_AVG": 16,
    "SI_DAY": 32,
    "SS_DAY": 28
}

# 6) 필요한 값만 추출
df_weather = pd.DataFrame({"DATE": df_weather_raw["DATE"]})

for key, idx in colmap.items():
    if idx < len(df_weather_raw.columns):  # 안전 장치
        df_weather[key] = pd.to_numeric(df_weather_raw[idx], errors='coerce')
    else:
        df_weather[key] = None

# 7) 원본 df와 merge
df = df.merge(df_weather, on="DATE", how="left")

C:\Users\hyebi\AppData\Local\Temp\ipykernel_35884\3535222913.py:8: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['DATE'] = pd.to_datetime(df['DATE'])


조회 범위: 20010324 ~ 20311025


In [6]:
df

,MTDT_INDEX,DTCT_UID,ROAD_POTHOLE,ROAD_CRACK,PLC_LTTD,PLC_LGTD,ROAD_INFO,DSTA_LGLD_INFO,DSTA_ADSTRD_INFO,IMG_CONF,...,TA_MIN,RN_DAY,RN_DUR,RN_D99,SD_DAY,SD_NEW,SD_MAX,HM_AVG,SI_DAY,SS_DAY
0,2603,RAD01_240119162357_441,1,0,36.384627,127.343459,대전광역시 유성구 신성동 산 26-12,신성동,신성동,NaN,...,1538.0,2.26,-9.0,1200.0,-9.0,-9.0,-9.0,0.0,9.3,853.0
1,2613,RAD01_240119162610_0,1,0,36.398713,127.357864,대전광역시 유성구 장동 202-11,장동,신성동,NaN,...,1538.0,2.26,-9.0,1200.0,-9.0,-9.0,-9.0,0.0,9.3,853.0
2,2614,RAD01_240119162611_0,1,0,36.398783,127.358005,대전광역시 유성구 장동 202-11,장동,신성동,NaN,...,1538.0,2.26,-9.0,1200.0,-9.0,-9.0,-9.0,0.0,9.3,853.0
3,2655,RAD01_240119163618_0,1,0,36.427899,127.387323,대전광역시 유성구 관평동 1286,관평동,관평동,NaN,...,1538.0,2.26,-9.0,1200.0,-9.0,-9.0,-9.0,0.0,9.3,853.0
4,2659,RAD01_240119163629_608,1,0,36.427741,127.387724,대전광역시 유성구 관평동 1286,관평동,관평동,NaN,...,1538.0,2.26,-9.0,1200.0,-9.0,-9.0,-9.0,0.0,9.3,853.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1489,23118,gnsoft1_251204085809_886916,1,0,36.390080,127.406226,대전광역시 유성구 문지동 667,문지동,전민동,0.8267,...,1214.0,1.91,-9.0,1200.0,-9.0,-9.0,-9.0,-0.9,5.7,1030.0
1490,23119,gnsoft1_251204085823_887180,1,0,36.390470,127.406918,대전광역시 유성구 문지동 668,문지동,전민동,0.8347,...,1214.0,1.91,-9.0,1200.0,-9.0,-9.0,-9.0,-0.9,5.7,1030.0
1491,23120,gnsoft1_251204085855_887652,1,0,36.391149,127.407477,대전광역시 유성구 문지동 676,문지동,전민동,0.8169,...,1214.0,1.91,-9.0,1200.0,-9.0,-9.0,-9.0,-0.9,5.7,1030.0
1492,23121,gnsoft1_251204085913_887841,1,0,36.390990,127.407868,대전광역시 유성구 문지동 661-3,문지동,전민동,0.8131,...,1214.0,1.91,-9.0,1200.0,-9.0,-9.0,-9.0,-0.9,5.7,1030.0


In [7]:
df.to_csv('data_daejeon2.csv', index=False, encoding='utf-8-sig')

## 교통량 데이터 통합

In [2]:
df = pd.read_csv("data_daejeon2.csv")
df.head()

,MTDT_INDEX,DTCT_UID,ROAD_POTHOLE,ROAD_CRACK,PLC_LTTD,PLC_LGTD,ROAD_INFO,DSTA_LGLD_INFO,DSTA_ADSTRD_INFO,IMG_CONF,...,TA_MIN,RN_DAY,RN_DUR,RN_D99,SD_DAY,SD_NEW,SD_MAX,HM_AVG,SI_DAY,SS_DAY
0,2603,RAD01_240119162357_441,1,0,36.384627,127.343459,대전광역시 유성구 신성동 산 26-12,신성동,신성동,NaN,...,1538.0,2.26,-9.0,1200.0,-9.0,-9.0,-9.0,0.0,9.3,853.0
1,2613,RAD01_240119162610_0,1,0,36.398713,127.357864,대전광역시 유성구 장동 202-11,장동,신성동,NaN,...,1538.0,2.26,-9.0,1200.0,-9.0,-9.0,-9.0,0.0,9.3,853.0
2,2614,RAD01_240119162611_0,1,0,36.398783,127.358005,대전광역시 유성구 장동 202-11,장동,신성동,NaN,...,1538.0,2.26,-9.0,1200.0,-9.0,-9.0,-9.0,0.0,9.3,853.0
3,2655,RAD01_240119163618_0,1,0,36.427899,127.387323,대전광역시 유성구 관평동 1286,관평동,관평동,NaN,...,1538.0,2.26,-9.0,1200.0,-9.0,-9.0,-9.0,0.0,9.3,853.0
4,2659,RAD01_240119163629_608,1,0,36.427741,127.387724,대전광역시 유성구 관평동 1286,관평동,관평동,NaN,...,1538.0,2.26,-9.0,1200.0,-9.0,-9.0,-9.0,0.0,9.3,853.0


In [3]:
url = "https://openapi.its.go.kr:9443/vdsInfo"
params = {
    "apiKey": "42eebed89dcb4a9f8fe2f2a4d97ecd76",
    "getType": "json"
}

headers = {
    "User-Agent": "Mozilla/5.0",
    "Host": "openapi.its.go.kr:9443",
    "Accept": "*/*",
    "Connection": "keep-alive"
}

res = requests.get(url, params=params, headers=headers)

# JSON 파일로 저장
with open("vds_info.json", "w", encoding="utf-8") as f:
    f.write(res.text)

print("✔ vds_info.json 저장 완료")

with open("vds_info.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# 구조 확인
# print(data.keys())        # header, body
# print(data["body"].keys()) # totalCount, items

# items 추출
items = data["body"]["items"]

# DataFrame 변환
df_temp = pd.DataFrame(items)

print(df_temp.head())
print("총 행수:", len(df_temp))

✔ vds_info.json 저장 완료
          vdsId                           linkIds laneNo     colctedDate  \
0  0010VDE00100  1400006000,1400007000,1400008700      1  20251224055800   
1  0010VDE00100  1400006000,1400007000,1400008700      2  20251224055800   
2  0010VDE00100  1400006000,1400007000,1400008700      3  20251224055800   
3  0010VDE00200             1400008700,1400011204      1  20251224055800   
4  0010VDE00200             1400008700,1400011204      2  20251224055800   

   speed volume occupancy  
0  107.5      8       1.5  
1     83      9       1.5  
2     69      4         1  
3     99      6         1  
4     90     14       2.5  
총 행수: 26235


In [12]:
API_KEY = "3415292884"

all_rows = []
page = 1

while True:
    params = {
        "key": API_KEY,
        "type": "json",
        "numOfRows": "5000",
        "pageNo": str(page)
    }
    
    res = requests.get("https://data.ex.co.kr/openapi/vdsinfo/vdsList", params=params)
    data = res.json()
    
    items = data.get("list", [])
    if not items:
        break
    
    all_rows.extend(items)
    
    # 페이지 정보
    pageSize = int(data["pageSize"]) if "pageSize" in data else None
    if pageSize and page >= pageSize:
        break
    
    page += 1

df_loc = pd.DataFrame(all_rows)
print("총 VDS 개수:", len(df_loc))

총 VDS 개수: 8493


In [13]:
print(df_loc.columns)
df_loc.head()

Index(['pageNo', 'numOfRows', 'routeNo', 'routeName', 'vdsId', 'shift',
       'vdsLength', 'roadgradeCode', 'roadgradeName', 'equipmentBelongingCode',
       'equipmentBelongingName', 'grs80x', 'grs80y', 'vdsStartShift',
       'vdsEndShift', 'vdsCode', 'vdsName', 'routeSeq', 'directionCode',
       'czId'],
      dtype='object')


,pageNo,numOfRows,routeNo,routeName,vdsId,shift,vdsLength,roadgradeCode,roadgradeName,equipmentBelongingCode,equipmentBelongingName,grs80x,grs80y,vdsStartShift,vdsEndShift,vdsCode,vdsName,routeSeq,directionCode,czId
0,None,None,0010,경부선,0010VDE00100,1.40km,1820.00m,101,고속국도,00,본선,391265.168637,297630.018235,0.20km,2.02km,1,FTMS 본선 VDS,1,E,0010CZE010
1,None,None,0010,경부선,0010VDE00200,2.40km,830.00m,101,고속국도,00,본선,391642.598589,298552.156704,2.02km,2.85km,1,FTMS 본선 VDS,2,E,0010CZE011
2,None,None,0010,경부선,0010VDE00300,3.30km,1160.00m,101,고속국도,00,본선,391873.613256,299421.826590,2.85km,4.01km,1,FTMS 본선 VDS,3,E,0010CZE011
3,None,None,0010,경부선,0010VDE00401,4.50km,1070.00m,101,고속국도,00,본선,391448.586328,300528.993655,4.01km,5.08km,1,FTMS 본선 VDS,4,E,0010CZE020
4,None,None,0010,경부선,0010VDE00500,5.10km,500.00m,101,고속국도,00,본선,391067.934813,300978.016235,5.08km,5.58km,1,FTMS 본선 VDS,5,E,0010CZE030


In [14]:
from pyproj import Transformer

# GRS80(중부원점) -> WGS84 변환기
transformer = Transformer.from_crs("EPSG:2097", "EPSG:4326", always_xy=True)

df_loc["lon"], df_loc["lat"] = transformer.transform(
    df_loc["grs80x"].astype(float),
    df_loc["grs80y"].astype(float)
)

print(df_loc[["vdsId", "lat", "lon"]].head())

          vdsId        lat         lon
0  0010VDE00100  36.160412  129.123371
1  0010VDE00200  36.168642  129.127787
2  0010VDE00300  36.176429  129.130565
3  0010VDE00401  36.186484  129.126114
4  0010VDE00500  36.190602  129.121994


In [16]:
def is_daejeon(lat, lon):
    return (
        36.176000 <= lat <= 36.502000 and
        127.185000 <= lon <= 127.487000
    )

df_loc["region"] = df_loc.apply(
    lambda r: "대전" if is_daejeon(r["lat"], r["lon"]) else "기타",
    axis=1
)

df_loc["region"].value_counts()

region
기타    8387
대전     106
Name: count, dtype: int64

In [19]:
df_merged = df_temp.merge(
    df_loc[["vdsId", "lat", "lon", "region"]],
    on="vdsId",
    how="left"
)

df_merged

,vdsId,linkIds,laneNo,colctedDate,speed,volume,occupancy,lat,lon,region
0,0010VDE00100,"1400006000,1400007000,1400008700",1,20251205144600,103.5,7,1,36.160412,129.123371,기타
1,0010VDE00100,"1400006000,1400007000,1400008700",2,20251205144600,84,13,2.5,36.160412,129.123371,기타
2,0010VDE00100,"1400006000,1400007000,1400008700",3,20251205144600,77,8,1,36.160412,129.123371,기타
3,0010VDE00200,"1400008700,1400011204",1,20251205144600,111,9,1.5,36.168642,129.127787,기타
4,0010VDE00200,"1400008700,1400011204",2,20251205144600,92.5,13,2.5,36.168642,129.127787,기타
...,...,...,...,...,...,...,...,...,...,...
25961,6000VDS03300,"1450593600,1450594500",2,20251205144600,78,23,11.8,36.184658,129.181558,기타
25962,6000VDS03400,"1450594500,1450595000,1450595100",1,20251205144600,101,14,7.42,36.181515,129.201168,기타
25963,6000VDS03400,"1450594500,1450595000,1450595100",2,20251205144600,84.5,6,3.78,36.181515,129.201168,기타
25964,6000VDS03500,"1450595100,1450595300,1450595500,1450596000",1,20251205144600,86,20,8.74,36.184490,129.225313,기타


In [21]:
df_daejeon = df_merged[df_merged["region"] == "대전"]
df_daejeon

,vdsId,linkIds,laneNo,colctedDate,speed,volume,occupancy,lat,lon,region
3674,0120VDE02400,"3100055700,3170033400,3170033600",1,20251205144600,-1,-1,-1,36.271979,127.196092,대전
3675,0120VDE02400,"3100055700,3170033400,3170033600",2,20251205144600,-1,-1,-1,36.271979,127.196092,대전
3676,0120VDE02500,3100055700,1,20251205144600,113,3,1.05,36.280616,127.216540,대전
3677,0120VDE02500,3100055700,2,20251205144600,108,5,2.82,36.280616,127.216540,대전
3678,0120VDE02520,"3100055700,3100055800,3100056000",1,20251205144600,-1,-1,-1,36.288896,127.235650,대전
...,...,...,...,...,...,...,...,...,...,...
9284,0270VDS03900,"3160031700,3160032600",2,20251205144600,100,0,0,36.456783,127.322248,대전
9285,0270VDS04200,"3160032600,3160032800",1,20251205144600,108.5,16,6.08,36.479829,127.323523,대전
9286,0270VDS04200,"3160032600,3160032800",2,20251205144600,113,4,1.24,36.479829,127.323523,대전
9287,0270VDS04400,"3160032700,3160032800",1,20251205144600,105.5,8,2.82,36.494651,127.327353,대전


In [24]:
from sklearn.neighbors import NearestNeighbors
import numpy as np

# 1) 포트홀 좌표 배열
pothole_coords = df[["PLC_LTTD", "PLC_LGTD"]].to_numpy()

# 2) VDS 좌표 배열
vds_coords = df_daejeon[["lat", "lon"]].to_numpy()

# 3) 모델 생성 (거리 기반 NN)
nn = NearestNeighbors(n_neighbors=1, algorithm='ball_tree')
nn.fit(vds_coords)

# 4) 각 포트홀에서 가장 가까운 VDS 찾기
distances, indices = nn.kneighbors(pothole_coords)

# 가장 가까운 VDS 행 번호
df["nearest_vds_index"] = indices.flatten()

In [27]:
df["nearest_vdsId"] = df["nearest_vds_index"].apply(lambda i: df_daejeon.iloc[i]["vdsId"])
df["speed"]         = df["nearest_vds_index"].apply(lambda i: df_daejeon.iloc[i]["speed"])
df["volume"]        = df["nearest_vds_index"].apply(lambda i: df_daejeon.iloc[i]["volume"])
df["occupancy"]     = df["nearest_vds_index"].apply(lambda i: df_daejeon.iloc[i]["occupancy"])

df

,MTDT_INDEX,DTCT_UID,ROAD_POTHOLE,ROAD_CRACK,PLC_LTTD,PLC_LGTD,ROAD_INFO,DSTA_LGLD_INFO,DSTA_ADSTRD_INFO,IMG_CONF,...,SD_NEW,SD_MAX,HM_AVG,SI_DAY,SS_DAY,nearest_vds_index,nearest_vdsId,speed,volume,occupancy
0,2603,RAD01_240119162357_441,1,0,36.384627,127.343459,대전광역시 유성구 신성동 산 26-12,신성동,신성동,NaN,...,-9.0,-9.0,0.0,9.3,853.0,134,0270VDS03505,95,8,2.76
1,2613,RAD01_240119162610_0,1,0,36.398713,127.357864,대전광역시 유성구 장동 202-11,장동,신성동,NaN,...,-9.0,-9.0,0.0,9.3,853.0,137,0270VDS03508,89.5,11,4.82
2,2614,RAD01_240119162611_0,1,0,36.398783,127.358005,대전광역시 유성구 장동 202-11,장동,신성동,NaN,...,-9.0,-9.0,0.0,9.3,853.0,137,0270VDS03508,89.5,11,4.82
3,2655,RAD01_240119163618_0,1,0,36.427899,127.387323,대전광역시 유성구 관평동 1286,관평동,관평동,NaN,...,-9.0,-9.0,0.0,9.3,853.0,147,0270VDS03900,100,0,0
4,2659,RAD01_240119163629_608,1,0,36.427741,127.387724,대전광역시 유성구 관평동 1286,관평동,관평동,NaN,...,-9.0,-9.0,0.0,9.3,853.0,147,0270VDS03900,100,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1489,23118,gnsoft1_251204085809_886916,1,0,36.390080,127.406226,대전광역시 유성구 문지동 667,문지동,전민동,0.8267,...,-9.0,-9.0,-0.9,5.7,1030.0,54,0120VDS03200,-1,-1,-1
1490,23119,gnsoft1_251204085823_887180,1,0,36.390470,127.406918,대전광역시 유성구 문지동 668,문지동,전민동,0.8347,...,-9.0,-9.0,-0.9,5.7,1030.0,54,0120VDS03200,-1,-1,-1
1491,23120,gnsoft1_251204085855_887652,1,0,36.391149,127.407477,대전광역시 유성구 문지동 676,문지동,전민동,0.8169,...,-9.0,-9.0,-0.9,5.7,1030.0,57,0120VDS03300,-1,-1,-1
1492,23121,gnsoft1_251204085913_887841,1,0,36.390990,127.407868,대전광역시 유성구 문지동 661-3,문지동,전민동,0.8131,...,-9.0,-9.0,-0.9,5.7,1030.0,57,0120VDS03300,-1,-1,-1


In [28]:
df.columns

Index(['MTDT_INDEX', 'DTCT_UID', 'ROAD_POTHOLE', 'ROAD_CRACK', 'PLC_LTTD',
       'PLC_LGTD', 'ROAD_INFO', 'DSTA_LGLD_INFO', 'DSTA_ADSTRD_INFO',
       'IMG_CONF', 'DATE', 'YEAR', 'MONTH', 'DAY', 'TA_AVG', 'TA_MAX',
       'TA_MIN', 'RN_DAY', 'RN_DUR', 'RN_D99', 'SD_DAY', 'SD_NEW', 'SD_MAX',
       'HM_AVG', 'SI_DAY', 'SS_DAY', 'nearest_vds_index', 'nearest_vdsId',
       'speed', 'volume', 'occupancy'],
      dtype='object')

In [31]:
temp_col = ['MTDT_INDEX', 'DTCT_UID', 'ROAD_INFO', 'DSTA_ADSTRD_INFO', 'IMG_CONF', 'nearest_vds_index', 'nearest_vdsId']

df.drop(columns=temp_col, inplace=True)

df

,ROAD_POTHOLE,ROAD_CRACK,PLC_LTTD,PLC_LGTD,DSTA_LGLD_INFO,DATE,YEAR,MONTH,DAY,TA_AVG,...,RN_D99,SD_DAY,SD_NEW,SD_MAX,HM_AVG,SI_DAY,SS_DAY,speed,volume,occupancy
0,1,0,36.384627,127.343459,신성동,2019-01-24,2024,1,19,1509.0,...,1200.0,-9.0,-9.0,-9.0,0.0,9.3,853.0,95,8,2.76
1,1,0,36.398713,127.357864,장동,2019-01-24,2024,1,19,1509.0,...,1200.0,-9.0,-9.0,-9.0,0.0,9.3,853.0,89.5,11,4.82
2,1,0,36.398783,127.358005,장동,2019-01-24,2024,1,19,1509.0,...,1200.0,-9.0,-9.0,-9.0,0.0,9.3,853.0,89.5,11,4.82
3,1,0,36.427899,127.387323,관평동,2019-01-24,2024,1,19,1509.0,...,1200.0,-9.0,-9.0,-9.0,0.0,9.3,853.0,100,0,0
4,1,0,36.427741,127.387724,관평동,2019-01-24,2024,1,19,1509.0,...,1200.0,-9.0,-9.0,-9.0,0.0,9.3,853.0,100,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1489,1,0,36.390080,127.406226,문지동,2004-12-25,2025,12,4,1302.0,...,1200.0,-9.0,-9.0,-9.0,-0.9,5.7,1030.0,-1,-1,-1
1490,1,0,36.390470,127.406918,문지동,2004-12-25,2025,12,4,1302.0,...,1200.0,-9.0,-9.0,-9.0,-0.9,5.7,1030.0,-1,-1,-1
1491,1,0,36.391149,127.407477,문지동,2004-12-25,2025,12,4,1302.0,...,1200.0,-9.0,-9.0,-9.0,-0.9,5.7,1030.0,-1,-1,-1
1492,1,0,36.390990,127.407868,문지동,2004-12-25,2025,12,4,1302.0,...,1200.0,-9.0,-9.0,-9.0,-0.9,5.7,1030.0,-1,-1,-1


In [32]:
df.to_csv("data_daejeon3.csv", index=False, encoding='utf-8-sig')

## 균열율 추가

In [33]:
df = pd.read_csv("data_daejeon3.csv")
df.head()

,ROAD_POTHOLE,ROAD_CRACK,PLC_LTTD,PLC_LGTD,DSTA_LGLD_INFO,DATE,YEAR,MONTH,DAY,TA_AVG,...,RN_D99,SD_DAY,SD_NEW,SD_MAX,HM_AVG,SI_DAY,SS_DAY,speed,volume,occupancy
0,1,0,36.384627,127.343459,신성동,2019-01-24,2024,1,19,1509.0,...,1200.0,-9.0,-9.0,-9.0,0.0,9.3,853.0,95.0,8,2.76
1,1,0,36.398713,127.357864,장동,2019-01-24,2024,1,19,1509.0,...,1200.0,-9.0,-9.0,-9.0,0.0,9.3,853.0,89.5,11,4.82
2,1,0,36.398783,127.358005,장동,2019-01-24,2024,1,19,1509.0,...,1200.0,-9.0,-9.0,-9.0,0.0,9.3,853.0,89.5,11,4.82
3,1,0,36.427899,127.387323,관평동,2019-01-24,2024,1,19,1509.0,...,1200.0,-9.0,-9.0,-9.0,0.0,9.3,853.0,100.0,0,0.00
4,1,0,36.427741,127.387724,관평동,2019-01-24,2024,1,19,1509.0,...,1200.0,-9.0,-9.0,-9.0,0.0,9.3,853.0,100.0,0,0.00


In [35]:
df.columns

Index(['ROAD_POTHOLE', 'ROAD_CRACK', 'PLC_LTTD', 'PLC_LGTD', 'DSTA_LGLD_INFO',
       'DATE', 'YEAR', 'MONTH', 'DAY', 'TA_AVG', 'TA_MAX', 'TA_MIN', 'RN_DAY',
       'RN_DUR', 'RN_D99', 'SD_DAY', 'SD_NEW', 'SD_MAX', 'HM_AVG', 'SI_DAY',
       'SS_DAY', 'speed', 'volume', 'occupancy'],
      dtype='object')

In [36]:
# === 0) 원본 df 보존 ===
df_result = df.copy()

# 1) volume / speed 기반 도로 등급 분류
q1, q2, q3 = df_result["volume"].quantile([0.33, 0.66, 0.9])

def classify_road(row):
    v = row["volume"]
    s = row["speed"]

    if (v > q3) and (s >= 70):
        return "highway"
    elif v > q2:
        return "arterial"
    elif v > q1:
        return "collector"
    else:
        return "local"

df_result["road_class"] = df_result.apply(classify_road, axis=1)

# 2) 도로 폭 매핑
width_map = {
    "highway": 20.0,
    "arterial": 15.0,
    "collector": 10.0,
    "local": 7.0
}
df_result["road_width_m"] = df_result["road_class"].map(width_map)

# 3) 도로 길이
df_result["road_len_m"] = 10.0

# 4) 포트홀 면적 생성
rng = np.random.default_rng(123)

def sample_pothole_area(row):
    if (row["ROAD_POTHOLE"] == 0) and (row["ROAD_CRACK"] == 0):
        return 0.0

    cls = row["road_class"]

    if cls == "local":
        low, high = 0.01, 0.08
    elif cls == "collector":
        low, high = 0.02, 0.12
    elif cls == "arterial":
        low, high = 0.03, 0.18
    else:
        low, high = 0.05, 0.25

    area = rng.uniform(low, high)

    if (row["ROAD_POTHOLE"] == 1) and (row["ROAD_CRACK"] == 1):
        area *= 1.3

    return round(area, 3)

df_result["pothole_area_m2"] = df_result.apply(sample_pothole_area, axis=1)

# 5) 균열율 계산
df_result["road_area_m2"] = df_result["road_width_m"] * df_result["road_len_m"]
df_result["crack_rate"] = (df_result["pothole_area_m2"] / df_result["road_area_m2"]) * 100

In [37]:
df_result

,ROAD_POTHOLE,ROAD_CRACK,PLC_LTTD,PLC_LGTD,DSTA_LGLD_INFO,DATE,YEAR,MONTH,DAY,TA_AVG,...,SS_DAY,speed,volume,occupancy,road_class,road_width_m,road_len_m,pothole_area_m2,road_area_m2,crack_rate
0,1,0,36.384627,127.343459,신성동,2019-01-24,2024,1,19,1509.0,...,853.0,95.0,8,2.76,collector,10.0,10.0,0.088,100.0,0.088000
1,1,0,36.398713,127.357864,장동,2019-01-24,2024,1,19,1509.0,...,853.0,89.5,11,4.82,arterial,15.0,10.0,0.038,150.0,0.025333
2,1,0,36.398783,127.358005,장동,2019-01-24,2024,1,19,1509.0,...,853.0,89.5,11,4.82,arterial,15.0,10.0,0.063,150.0,0.042000
3,1,0,36.427899,127.387323,관평동,2019-01-24,2024,1,19,1509.0,...,853.0,100.0,0,0.00,collector,10.0,10.0,0.038,100.0,0.038000
4,1,0,36.427741,127.387724,관평동,2019-01-24,2024,1,19,1509.0,...,853.0,100.0,0,0.00,collector,10.0,10.0,0.038,100.0,0.038000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1489,1,0,36.390080,127.406226,문지동,2004-12-25,2025,12,4,1302.0,...,1030.0,-1.0,-1,-1.00,local,7.0,10.0,0.061,70.0,0.087143
1490,1,0,36.390470,127.406918,문지동,2004-12-25,2025,12,4,1302.0,...,1030.0,-1.0,-1,-1.00,local,7.0,10.0,0.037,70.0,0.052857
1491,1,0,36.391149,127.407477,문지동,2004-12-25,2025,12,4,1302.0,...,1030.0,-1.0,-1,-1.00,local,7.0,10.0,0.018,70.0,0.025714
1492,1,0,36.390990,127.407868,문지동,2004-12-25,2025,12,4,1302.0,...,1030.0,-1.0,-1,-1.00,local,7.0,10.0,0.056,70.0,0.080000


In [48]:
df_result["crack_rate"].describe()

count    1494.000000
mean        0.068114
std         0.028650
min         0.014286
25%         0.043333
50%         0.067500
75%         0.092667
max         0.124500
Name: crack_rate, dtype: float64

In [49]:
df_result["pothole_area_m2"].describe()

count    1494.000000
mean        0.076027
std         0.047505
min         0.010000
25%         0.041000
50%         0.065000
75%         0.098000
max         0.249000
Name: pothole_area_m2, dtype: float64

In [50]:
df_result.to_csv("data_daejeon_final.csv", index=False, encoding='utf-8-sig')